In [1]:
!pip install \
sentence-transformers \
faiss-cpu \
pypdf \
transformers \
torch \
accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 12.3 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [18]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print(pdf_file)

Saving cloud_security_research.pdf to cloud_security_research.pdf
cloud_security_research.pdf


In [19]:
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

Cloud Security Challenges and Solutions in Modern Computing Environments 
Abstract 
Cloud computing has revolutionized information technology by providing scalable, flexible, 
and cost-effective computing resources. Organizations increasingly rely on cloud 
platforms to store data, host applications, and perform large-scale computations. Despite 
its advantages, cloud computing introduces several security concerns such as 
unauthorized access, data breaches, insider threats, and misconfigured resources. 
This study investigates major cloud security challenges and evaluates existing mitigation 
strategies. Security mechanisms including encryption, multi-factor authentication, identity 
and access management, and intrusion detection systems are analyzed. The research also 
explores emerging technologies such as artificial intelligence and blockchain for improving 
cloud security. 
Introduction 
Cloud computing enables users to access computing resources over the internet. Public 
cloud p

In [20]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):

    chunks.append(
        text[i:i+chunk_size]
    )

print("Chunks:", len(chunks))

Chunks: 7


In [21]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = model.encode(
    chunks
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(7, 384)

In [23]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

print(
    "Vectors Stored:",
    index.ntotal
)

Vectors Stored: 7


In [24]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [27]:
query = input(
    "Ask Research Question: "
)

Ask Research Question: What are the major cloud security challenges?


In [28]:
query_embedding = model.encode(
    [query]
)

D, I = index.search(
    np.array(query_embedding),
    k=3
)

context = ""

for idx in I[0]:

    context += chunks[idx]

print(context[:1000])

roviders offer services such as Infrastructure as a Service (IaaS), Platform as a 
Service (PaaS), and Software as a Service (SaaS). 
Although cloud environments provide flexibility and scalability, they expose organizations 
to multiple security risks. Sensitive information stored in cloud infrastructure may become 
vulnerable to cyberattacks if proper security measures are not implemented. 
Cloud Security Challenges 
The major challenges include: 
1. Data Privacy  
2. Unauthorized Access  
3. Cloud Security Challenges and Solutions in Modern Computing Environments 
Abstract 
Cloud computing has revolutionized information technology by providing scalable, flexible, 
and cost-effective computing resources. Organizations increasingly rely on cloud 
platforms to store data, host applications, and perform large-scale computations. Despite 
its advantages, cloud computing introduces several security concerns such as 
unauthorized access, data breaches, insider threats, and misconfigured re

In [29]:
prompt = f"""
Context:

{context}

Question:

{query}

Answer:
"""

answer = generator(
    prompt,
    max_length=256
)

print(
    answer[0]["generated_text"]
)

Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:

roviders offer services such as Infrastructure as a Service (IaaS), Platform as a 
Service (PaaS), and Software as a Service (SaaS). 
Although cloud environments provide flexibility and scalability, they expose organizations 
to multiple security risks. Sensitive information stored in cloud infrastructure may become 
vulnerable to cyberattacks if proper security measures are not implemented. 
Cloud Security Challenges 
The major challenges include: 
1. Data Privacy  
2. Unauthorized Access  
3. Cloud Security Challenges and Solutions in Modern Computing Environments 
Abstract 
Cloud computing has revolutionized information technology by providing scalable, flexible, 
and cost-effective computing resources. Organizations increasingly rely on cloud 
platforms to store data, host applications, and perform large-scale computations. Despite 
its advantages, cloud computing introduces several security concerns such as 
unauthorized access, data breaches, insider threats, and misco

In [30]:
summary_prompt = f"""
Summarize this research paper:

{text[:4000]}
"""

summary = generator(
    summary_prompt,
    max_length=300
)

print(
    summary[0]["generated_text"]
)

Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Summarize this research paper:

Cloud Security Challenges and Solutions in Modern Computing Environments 
Abstract 
Cloud computing has revolutionized information technology by providing scalable, flexible, 
and cost-effective computing resources. Organizations increasingly rely on cloud 
platforms to store data, host applications, and perform large-scale computations. Despite 
its advantages, cloud computing introduces several security concerns such as 
unauthorized access, data breaches, insider threats, and misconfigured resources. 
This study investigates major cloud security challenges and evaluates existing mitigation 
strategies. Security mechanisms including encryption, multi-factor authentication, identity 
and access management, and intrusion detection systems are analyzed. The research also 
explores emerging technologies such as artificial intelligence and blockchain for improving 
cloud security. 
Introduction 
Cloud computing enables users to access computing resources o

In [32]:
from collections import Counter
import re

words = re.findall(
    r'\b[a-zA-Z]{5,}\b',
    text.lower()
)

common = Counter(words)

print(
    common.most_common(20)
)

[('cloud', 18), ('security', 18), ('computing', 9), ('challenges', 7), ('access', 6), ('environments', 5), ('artificial', 5), ('intelligence', 5), ('solutions', 4), ('modern', 4), ('systems', 4), ('research', 4), ('blockchain', 4), ('service', 4), ('information', 3), ('resources', 3), ('existing', 3), ('encryption', 3), ('multi', 3), ('authentication', 3)]


In [33]:
gap_prompt = f"""
Based on this research,
identify future work and research gaps.

{text[:4000]}
"""

gaps = generator(
    gap_prompt,
    max_length=300
)

print(
    gaps[0]["generated_text"]
)

Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Based on this research,
identify future work and research gaps.

Cloud Security Challenges and Solutions in Modern Computing Environments 
Abstract 
Cloud computing has revolutionized information technology by providing scalable, flexible, 
and cost-effective computing resources. Organizations increasingly rely on cloud 
platforms to store data, host applications, and perform large-scale computations. Despite 
its advantages, cloud computing introduces several security concerns such as 
unauthorized access, data breaches, insider threats, and misconfigured resources. 
This study investigates major cloud security challenges and evaluates existing mitigation 
strategies. Security mechanisms including encryption, multi-factor authentication, identity 
and access management, and intrusion detection systems are analyzed. The research also 
explores emerging technologies such as artificial intelligence and blockchain for improving 
cloud security. 
Introduction 
Cloud computing enables user